In [ ]:
%pip install sb3-contrib

In [ ]:
from stable_baselines3 import DQN
from sb3_contrib import RecurrentPPO
import json
import cat_toy_env
import gym_agents

vel_seq_len = 10  # 速度履歴長を定数で定義

with open("../cat-game/public/common.json") as f:
    env_config = json.load(f)

"""
key_wards_eval["runners"] = [
        (lambda: gym_agents.RunnerEscape(vel_seq_len=vel_seq_len, energy=1000, speed=0.5), 1),
        (lambda: gym_agents.RunnerCircle(vel_seq_len=vel_seq_len, energy=1000, speed=0.5, radius=50.0), 1),
        (lambda: gym_agents.RunnerRandom(vel_seq_len=vel_seq_len, energy=-1000, speed=0.5), 1),
        (lambda: gym_agents.RunnerEscapeWhenTooClose(vel_seq_len=vel_seq_len, energy=1000, threshold=0.2, speed=0.5), 1),
        (lambda: gym_agents.RunnerStop(vel_seq_len=vel_seq_len, energy=1000), 1),
        (lambda: gym_agents.RunnerStopAndMove(speed=0.5, move_interval=10, vel_seq_len=vel_seq_len, energy=1000), 1),
        (lambda: gym_agents.RunnerSnake(vel_seq_len=vel_seq_len, energy=1000, speed=0.5, amplitude=1.0, freq=1.0), 1),
        (lambda: gym_agents.RunnerOscillation(vel_seq_len=vel_seq_len, energy=1000, axis='x', amplitude=10.0, freq=1/10), 1),
        (lambda: gym_agents.Enemy(vel_seq_len=vel_seq_len, energy=1000, speed=0.5), 1)
]
"""

key_wards = {
    "render_mode": "",
    "config": env_config,
    "max_steps": 1000,
    "chaser": lambda: gym_agents.cat.Cat(),
    "runners": [
        (lambda: gym_agents.RunnerEscape(vel_seq_len=vel_seq_len, energy=-1000, speed=0.5), 0.25),
        (lambda: gym_agents.RunnerCircle(vel_seq_len=vel_seq_len, energy=1000, speed=0.5, radius=50.0), 0.5),
        (lambda: gym_agents.RunnerRandom(vel_seq_len=vel_seq_len, energy=-1000, speed=0.5), 0.25),
    ],
    "reset_interval": 2000
}

env = cat_toy_env.CatToyEnv(**key_wards)

model = DQN("MlpPolicy", env, verbose=1)


In [ ]:
import importlib
importlib.reload(cat_toy_env)
importlib.reload(gym_agents)

## モデルをロード（必要な場合のみ）

In [ ]:
model = DQN.load("models/sb/cat")
model.set_env(env)  # 必要に応じて新しい環境をセット

## 学習

In [ ]:
model.learn(total_timesteps=150000, log_interval=4)
model.save("models/sb/cat")  # 上書き保存

In [ ]:
import numpy as np

key_wards_eval = key_wards.copy()
key_wards_eval["render_mode"] = "human"

env_eval = cat_toy_env.CatToyEnv(**key_wards_eval)

model = DQN.load("models/sb/cat")

obs, _ = env_eval.reset()
# cell and hidden state of the LSTM
lstm_states = None
num_envs = 1
# Episode start signals are used to reset the lstm states
episode_starts = np.ones((num_envs,), dtype=bool)
done = False
while not done:
    action, lstm_states = model.predict(obs, deterministic=True)
    obs, rewards, terminated, truncated, info = env_eval.step(action)
    done = terminated or truncated

In [ ]:
import torch
import dqn_onnx
import importlib
importlib.reload(dqn_onnx)

# 7次元のダミー入力（相対位置2, chaser速度2, runner速度2*seq_len, fatigue1）
obs = torch.randn(1, 2+2+2*vel_seq_len+1).detach().cpu()
#concat_input = obs.repeat(config["cat"]["dqn"]["rnn"]["sequence_length"], 1).unsqueeze(0)  # shape: (1, sequence_length, 7)

# モデルのロード
policy_net = dqn_onnx.DQNOnnx(model.policy)

# ONNX エクスポート
torch.onnx.export(
    policy_net,
    (obs),
    "cat_dqn_policy.onnx",
    export_params=True,
    opset_version=17,
    input_names=["obs"],
    output_names=["option", "action"],
    dynamic_axes={
        "obs": {0: "batch_size"},  # 観測データのバッチ次元を可変に
        "option": {0: "batch_size"},
        "action": {0: "batch_size"},
    },
    training=torch.onnx.TrainingMode.EVAL
)